# 02_train_baseline_model (NACC) — Updated

Trains a simple baseline 4-class classifier using `output/cohort.csv` produced by `01_build_cohort`.


In [26]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier


OUT_DIR  = os.path.join("..", "output")
DATA_PATH = os.path.join(OUT_DIR, "cohort.csv")

df = pd.read_csv(DATA_PATH)

print("Loaded:", df.shape)
print(df["label"].value_counts(dropna=False))
df.head()


Loaded: (37007, 55)
label
Censored         22339
NoDementia_5y    10816
AD                2948
MixedOrOther       445
PD_LBD             283
Vascular           176
Name: count, dtype: int64


,NACCID,baseline_date,NACCAGE,SEX,RACE,HISPANIC,EDUC,DIABETES,HYPERTEN,HYPERCHO,...,BPSYS_missing,BPDIAS_missing,STROKCOG_missing,CVDCOG_missing,STKIMAG_missing,CVDIMAG_missing,CSFABETA,CSFPTAU,CSFTTAU,abs_days_from_baseline
0,NACC000011,2006-04-17,62,2,1,0,16.0,0,0,0,...,0,0,1,1,1,1,NaN,NaN,NaN,NaN
1,NACC000034,2015-07-16,79,2,1,0,15.0,0,1,0,...,0,0,1,1,0,1,NaN,NaN,NaN,NaN
2,NACC000067,2012-05-23,60,1,1,0,18.0,0,0,0,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN
3,NACC000073,2016-05-20,44,2,1,0,18.0,0,0,0,...,1,0,1,1,0,1,NaN,NaN,NaN,NaN
4,NACC000144,2010-08-17,79,1,1,0,18.0,0,1,1,...,0,0,1,0,1,1,NaN,NaN,NaN,NaN


## Keep only the 4-class task

In [27]:
KEEP = {"AD", "Vascular", "PD_LBD", "NoDementia_5y"}
df4 = df[df["label"].isin(KEEP)].copy()

print("4-class rows:", len(df4))
print(df4["label"].value_counts())


4-class rows: 14223
label
NoDementia_5y    10816
AD                2948
PD_LBD             283
Vascular           176
Name: count, dtype: int64


## Split + preprocess + train

In [28]:
y = df4["label"].astype(str)

drop_cols = [c for c in ["NACCID", "baseline_date", "label"] if c in df4.columns]
X = df4.drop(columns=drop_cols)

MISSING_CODES = {-4, 9}
RACE_MISSING = {99}  # from dictionary screenshot

# 1) Demographics as categorical
for c in ["SEX", "HISPANIC", "RACE"]:
    if c in X.columns:
        X[c] = pd.to_numeric(X[c], errors="coerce")

if "RACE" in X.columns:
    X["RACE"] = X["RACE"].mask(X["RACE"].isin(list(RACE_MISSING)), np.nan)

for c in ["SEX", "HISPANIC", "RACE"]:
    if c in X.columns:
        X[c] = X[c].mask(X[c].isin(list(MISSING_CODES)), np.nan)
        X[c] = X[c].astype("Int64").astype(str)   # becomes categorical for OneHot

# 2) Medical conditions → binary "ever present"
#COND_VARS = ["DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA"]
COND_VARS = [
    "DIABETES","HYPERTEN","HYPERCHO","CVAFIB","CBSTROKE","CBTIA","APNEA",
    "CVCHF","CVANGINA","CVHATT","PREVSTK","STROKCOG","CVDCOG","STKIMAG",
    "CVDIMAG","CVDIMAG1","CVDIMAG2","CVDIMAG3","CVDIMAG4"
]

for c in COND_VARS:
    if c in X.columns:
        s = pd.to_numeric(X[c], errors="coerce")
        s = s.mask(s.isin(list(MISSING_CODES)), np.nan)
        X[c] = s.map(lambda v: 1 if v in [1,2] else (0 if v==0 else np.nan))

# 3) DEP2YRS as binary
if "DEP2YRS" in X.columns:
    s = pd.to_numeric(X["DEP2YRS"], errors="coerce")
    s = s.mask(s.isin(list(MISSING_CODES)), np.nan)
    X["DEP2YRS"] = s.map(lambda v: 1 if v==1 else (0 if v==0 else np.nan))


cat_cols = [c for c in X.columns if X[c].dtype == "object"]
num_cols = [c for c in X.columns if c not in cat_cols]

print("Num cols:", len(num_cols), "Cat cols:", len(cat_cols))

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

numeric_pipe = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_pipe_sparse = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))  # sparse default
])

categorical_pipe_dense = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False))
])

preprocess_sparse = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe_sparse, cat_cols),
    ],
    remainder="drop"
)

preprocess_dense = ColumnTransformer(
    transformers=[
        ("num", numeric_pipe, num_cols),
        ("cat", categorical_pipe_dense, cat_cols),
    ],
    remainder="drop"
)

#clf_logreg = Pipeline(steps=[
#    ("preprocess", preprocess_sparse),
#    ("model", LogisticRegression(
#        max_iter=4000,
#        class_weight="balanced"
#    ))
#])

custom_w = {
    "NoDementia_5y": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 8.0
}

clf_logreg = Pipeline(steps=[
    ("preprocess", preprocess_sparse),
    ("model", LogisticRegression(
        max_iter=6000,
        class_weight=custom_w
    ))
])


clf_tree = Pipeline(steps=[
    ("preprocess", preprocess_dense),
    ("model", HistGradientBoostingClassifier(
        max_depth=6,
        learning_rate=0.05,
        max_iter=300,
        random_state=42
    ))
])

clf_tree_tuned = Pipeline(steps=[
    ("preprocess", preprocess_dense),
    ("model", HistGradientBoostingClassifier(
        max_depth=3,          # smaller depth can generalize better
        learning_rate=0.05,
        max_iter=800,         # more trees
        min_samples_leaf=20,  # reduce overfitting
        l2_regularization=0.1,
        random_state=42
    ))
])



Num cols: 49 Cat cols: 3


In [29]:
X.dtypes.sort_values().head(50)

has_5y_followup              bool
NACCAGE                     int64
BOSTON                      int64
ANIMALS                     int64
TRAILB_missing              int64
NACCMMSE_missing            int64
NACCMOCA_missing            int64
CDRSUM_missing              int64
TRAILA_missing              int64
BPDIAS_missing              int64
STROKCOG_missing            int64
CVDCOG_missing              int64
STKIMAG_missing             int64
CVDIMAG_missing             int64
BPSYS_missing               int64
STKIMAG                   float64
CVDIMAG                   float64
CVDIMAG1                  float64
CVDIMAG2                  float64
CSFABETA                  float64
CSFPTAU                   float64
CVDIMAG4                  float64
max_follow_years          float64
CVDCOG                    float64
CVDIMAG3                  float64
STROKCOG                  float64
CVHATT                    float64
CSFTTAU                   float64
EDUC                      float64
DIABETES      

## Evaluate

In [30]:
from sklearn.metrics import classification_report, confusion_matrix, balanced_accuracy_score
from sklearn.utils.class_weight import compute_sample_weight

labels = sorted(list(KEEP))

# --- Logistic regression (balanced) ---
clf_logreg.fit(X_train, y_train)
pred_lr = clf_logreg.predict(X_test)

# 1) Best overall stability (and simplest baseline)

# Logistic Regression (custom weights)
# Macro-F1 ~ 0.50, strong AD + NoDementia.



print("=== Logistic Regression (custom class weights) ===")
print(classification_report(y_test, pred_lr, digits=4))
print("Confusion matrix:", labels)
print(confusion_matrix(y_test, pred_lr, labels=labels))
print()

# --- Tree model ---
# 2) Best rare-class sensitivity (especially vascular)

# Tree tuned + sample_weight="balanced"
# This one had Vascular recall ~0.37 and PD_LBD recall ~0.53, and balanced accuracy ~0.58 (your best balanced accuracy so far).


w_train = compute_sample_weight(class_weight="balanced", y=y_train)
# Custom class weights (same idea as LR)
# custom_w = {
#     "NoDementia_5y": 1.0,
#     "AD": 2.0,
#     "PD_LBD": 6.0,
#     "Vascular": 8.0
# }

# Build sample_weight vector for training rows
# w_train = y_train.map(custom_w).astype(float).values

# clf_tree_tuned.fit(X_train, y_train, model__sample_weight=w_train)
# pred_tree_custom = clf_tree_tuned.predict(X_test)

# print("=== HistGradientBoosting tuned (csample_weight="balanced") ===")
# print(classification_report(y_test, pred_tree_custom, digits=4))
# print(confusion_matrix(y_test, pred_tree_custom, labels=labels))
# print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_tree_custom))


clf_tree_tuned.fit(X_train, y_train, model__sample_weight=w_train)
pred_tree_w = clf_tree_tuned.predict(X_test)

print("=== HistGradientBoosting tuned (sample_weight=balanced) ===")
print(classification_report(y_test, pred_tree_w, digits=4))
print(confusion_matrix(y_test, pred_tree_w, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_tree_w))



=== Logistic Regression (custom class weights) ===
               precision    recall  f1-score   support

           AD     0.7103    0.7271    0.7186       590
NoDementia_5y     0.9462    0.9427    0.9444      2163
       PD_LBD     0.2549    0.2281    0.2407        57
     Vascular     0.2286    0.2286    0.2286        35

     accuracy                         0.8749      2845
    macro avg     0.5350    0.5316    0.5331      2845
 weighted avg     0.8746    0.8749    0.8747      2845

Confusion matrix: ['AD', 'NoDementia_5y', 'PD_LBD', 'Vascular']
[[ 429  107   35   19]
 [ 117 2039    2    5]
 [  37    4   13    3]
 [  21    5    1    8]]

=== HistGradientBoosting tuned (sample_weight=balanced) ===
               precision    recall  f1-score   support

           AD     0.6496    0.5153    0.5747       590
NoDementia_5y     0.9652    0.8988    0.9308      2163
       PD_LBD     0.1627    0.5965    0.2556        57
     Vascular     0.0844    0.3714    0.1376        35

     accura

In [32]:
from sklearn.model_selection import GridSearchCV

param_grid = {"model__C": [0.1, 0.3, 1.0, 3.0, 10.0]}

lr = Pipeline(steps=[
    ("preprocess", preprocess_sparse),
    ("model", LogisticRegression(max_iter=8000, class_weight=custom_w))
])

gs = GridSearchCV(lr, param_grid, scoring="f1_macro", cv=3, n_jobs=-1)
gs.fit(X_train, y_train)

print("Best params:", gs.best_params_)
best_lr = gs.best_estimator_
pred = best_lr.predict(X_test)

print("=== Logistic Regression tuned C (custom weights) ===")
print(classification_report(y_test, pred, digits=4))
print(confusion_matrix(y_test, pred, labels=labels))

Best params: {'model__C': 0.1}
=== Logistic Regression tuned C (custom weights) ===
               precision    recall  f1-score   support

           AD     0.7155    0.7288    0.7221       590
NoDementia_5y     0.9458    0.9441    0.9449      2163
       PD_LBD     0.2600    0.2281    0.2430        57
     Vascular     0.2286    0.2286    0.2286        35

     accuracy                         0.8763      2845
    macro avg     0.5375    0.5324    0.5346      2845
 weighted avg     0.8755    0.8763    0.8758      2845

[[ 430  108   34   18]
 [ 113 2042    2    6]
 [  37    4   13    3]
 [  21    5    1    8]]


In [31]:
# 3) Best compromise (decent AD + some vascular)

# Tree tuned + custom weights with Vascular=20
# Macro-F1 ~ 0.53 and balanced accuracy ~0.54.


custom_w2 = {
    "NoDementia_5y": 1.0,
    "AD": 2.0,
    "PD_LBD": 6.0,
    "Vascular": 20.0   # increase a lot
}
w_train2 = y_train.map(custom_w2).astype(float).values

clf_tree_tuned.fit(X_train, y_train, model__sample_weight=w_train2)
pred_tree_custom2 = clf_tree_tuned.predict(X_test)

print("=== Tree tuned (custom sample_weight) ===")
print(classification_report(y_test, pred_tree_custom2, digits=4))
print(confusion_matrix(y_test, pred_tree_custom2, labels=labels))
print("Balanced accuracy:", balanced_accuracy_score(y_test, pred_tree_custom2))


=== Tree tuned (custom sample_weight) ===
               precision    recall  f1-score   support

           AD     0.7320    0.7220    0.7270       590
NoDementia_5y     0.9542    0.9450    0.9496      2163
       PD_LBD     0.3265    0.2807    0.3019        57
     Vascular     0.1250    0.2571    0.1682        35

     accuracy                         0.8770      2845
    macro avg     0.5344    0.5512    0.5367      2845
 weighted avg     0.8854    0.8770    0.8808      2845

[[ 426   90   25   49]
 [ 102 2044    8    9]
 [  32    4   16    5]
 [  22    4    0    9]]
Balanced accuracy: 0.5512155821510334
